# Part II: A* Search Algorithm for City Navigation

**Objective:** Find the shortest path, total cost, and intermediate cities between any two cities on the given map.

<br><br>

The weighted map is represented as an undirected adjacency dictionary. A* evaluates each candidate city using

<br>

$$f(n)=g(n)+h(n),$$

<br>

where $g(n)$ is the confirmed cost from the start and $h(n)$ is a safe lower-bound estimate to the goal.

## 1. Graph Representation

Each city maps to its directly connected neighbours and the corresponding edge costs. Every map connection is stored in both directions because travel is undirected.

In [8]:
from collections import deque
import heapq
import math


graph = {
    "Atlanta": {"Dallas": 721, "Houston": 702, "Chicago": 588, "Washington": 543, "Miami": 604},
    "Boston": {"Detroit": 613, "New York": 190},
    "Chicago": {"Seattle": 1737, "Riverside": 1704, "Dallas": 805, "Atlanta": 588, "Detroit": 238},
    "Dallas": {"Phoenix": 887, "Chicago": 805, "Houston": 225, "Atlanta": 721},
    "Detroit": {"Chicago": 238, "Boston": 613, "New York": 482, "Washington": 396},
    "Houston": {"Phoenix": 1015, "Dallas": 225, "Atlanta": 702, "Miami": 968},
    "Los Angeles": {"San Francisco": 348, "Riverside": 50, "Phoenix": 357},
    "Miami": {"Houston": 968, "Washington": 923, "Atlanta": 604},
    "New York": {"Detroit": 482, "Boston": 190, "Philadelphia": 81},
    "Philadelphia": {"New York": 81, "Washington": 123},
    "Phoenix": {"Los Angeles": 357, "Riverside": 307, "Dallas": 887, "Houston": 1015},
    "Riverside": {"San Francisco": 386, "Los Angeles": 50, "Phoenix": 307, "Chicago": 1704},
    "San Francisco": {"Seattle": 678, "Los Angeles": 348, "Riverside": 386},
    "Seattle": {"San Francisco": 678, "Chicago": 1737},
    "Washington": {"Detroit": 396, "Philadelphia": 123, "Atlanta": 543, "Miami": 923},
}

## 2. A* Implementation

Since the map does not provide geographic coordinates or separate heuristic values. Therefore, the heuristic uses only valid information from the supplied graph:

$$h(n)=L(n)\times c_{\min},$$

where $L(n)$ is the minimum number of remaining edges to the goal and $c_{\min}$ is the smallest edge cost in the map. Any route must use at least $L(n)$ edges, and every edge costs at least $c_{\min}$. Therefore, $h(n)$ never overestimates the true remaining cost. It is also consistent, so the goal is optimal when removed from the priority queue.

In [9]:
def validate_graph(graph):
    if not graph:
        raise ValueError("The graph cannot be empty.")

    for city, neighbours in graph.items():
        for neighbour, cost in neighbours.items():
            if neighbour not in graph:
                raise ValueError(f"Unknown city referenced: {neighbour}")
            if not isinstance(cost, (int, float)) or not math.isfinite(cost) or cost <= 0:
                raise ValueError(f"Invalid cost on {city} -> {neighbour}: {cost}")
            if graph[neighbour].get(city) != cost:
                raise ValueError(f"Asymmetric edge: {city} <-> {neighbour}")


def hop_distance_to_goal(graph, goal):
    hop_distance = {goal: 0}
    queue = deque([goal])

    while queue:
        current = queue.popleft()
        for neighbour in graph[current]:
            if neighbour not in hop_distance:
                hop_distance[neighbour] = hop_distance[current] + 1
                queue.append(neighbour)

    return hop_distance


def make_heuristic(graph, goal):
    minimum_edge_cost = min(
        cost
        for neighbours in graph.values()
        for cost in neighbours.values()
    )

    hop_distance = hop_distance_to_goal(graph, goal)

    def heuristic(city):
        return hop_distance.get(city, 0) * minimum_edge_cost

    return heuristic


def reconstruct_path(parent, start, goal):
    path = [goal]

    while path[-1] != start:
        path.append(parent[path[-1]])

    return list(reversed(path))


def path_cost(graph, path):
    return sum(graph[current][next_city] for current, next_city in zip(path, path[1:]))


def a_star(graph, start, goal):
    if start not in graph:
        raise ValueError(f"Unknown start city: {start}")
    if goal not in graph:
        raise ValueError(f"Unknown goal city: {goal}")

    heuristic = make_heuristic(graph, goal)
    g_score = {city: math.inf for city in graph}
    g_score[start] = 0
    parent = {}
    open_queue = [(heuristic(start), 0, start)]

    while open_queue:
        f_n, current_cost, current = heapq.heappop(open_queue)

        if current_cost != g_score[current]:
            continue
        if current == goal:
            return reconstruct_path(parent, start, goal), current_cost

        for neighbour, edge_cost in graph[current].items():
            tentative_cost = current_cost + edge_cost
            if tentative_cost < g_score[neighbour]:
                g_score[neighbour] = tentative_cost
                parent[neighbour] = current
                estimated_total = tentative_cost + heuristic(neighbour)
                heapq.heappush(
                    open_queue, (estimated_total, tentative_cost, neighbour)
                )

    raise ValueError(f"No path exists from {start} to {goal}.")


validate_graph(graph)

## 3. Example Shortest Path

Only `START_CITY` and `GOAL_CITY` need to be changed to test another valid pair of cities.

In [10]:
START_CITY = "Seattle"
GOAL_CITY = "Miami"

shortest_path, total_cost = a_star(graph, START_CITY, GOAL_CITY)
intermediate_cities = shortest_path[1 : -1]

print("Shortest path:", " -> ".join(shortest_path))
print("Total cost:", total_cost)
print("Intermediate cities:", ", ".join(intermediate_cities) or "None")
print("Cities along the path:", len(shortest_path))

Shortest path: Seattle -> Chicago -> Atlanta -> Miami
Total cost: 2929
Intermediate cities: Chicago, Atlanta
Cities along the path: 4


## 4. Output Result

For Example: A* returns **Seattle → Chicago → Atlanta → Miami** with a total cost of

<br>

$$1737+588+604=2929.$$

<br>

The priority queue always expands the city with the smallest estimated total cost $f(n)$. When a cheaper route to a city is found, its best known cost and parent are updated. Because the heuristic is admissible and consistent, the returned route is guaranteed to be the shortest.